# 04 Graph — Building Graphs from OBJ Geometry

Synthesises **S02-01 → S02-06** graph concepts applied to the actual Rhino building model.

| Section | Concept | S02 Reference |
|---------|---------|---------------|
| 1 | OBJ Import | S02-06 |
| 2 | CellComplex | — |
| 3 | Primal vs Dual | S02-01 

In [11]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
import colorsys

print(Helper.Version())
renderer = 'vscode'

GEO  = 'c:/Users/etmaglari/IAAC/etmaglari_gML/Rhino_Geometry _etm.obj'
APTS = 'c:/Users/etmaglari/IAAC/etmaglari_gML/Rhino_Apertures _etm.obj'

The version that you are using (0.9.18) is OLDER than the latest version (0.9.21) from PyPI. Please consider upgrading to the latest version.


## 1. OBJ Geometry  (→ S02-06)

Import the building shell and apertures exported from Rhino.

In [12]:
geo_objects = Topology.ByOBJPath(GEO)
print(f'Geometry: {len(geo_objects)} object(s)')

Topology.Show(geo_objects,
              faceColor=[210, 210, 250],
              faceOpacity=0.6,
              edgeColor='white',
              edgeWidth=2,
              showVertices=False,
              backgroundColor='black',
              width=800, height=600,
              renderer=renderer)

Geometry: 1 object(s)


In [13]:
apt_objects = Topology.ByOBJPath(APTS)
print(f'Apertures: {len(apt_objects)} object(s)')

Topology.Show(apt_objects,
              faceColor=[255, 200, 80],
              faceOpacity=0.9,
              edgeColor='white',
              edgeWidth=2,
              showVertices=False,
              backgroundColor='black',
              width=800, height=600,
              renderer=renderer)

Topology.ByOBJPath - Error: the input OBJ path does not exist. Returning None.


TypeError: object of type 'NoneType' has no len()

## 2. Build CellComplex

The OBJ shell captures geometry only. Rooms are reconstructed programmatically from
the same floor-plan corners so topological graph operations can be applied.

In [ ]:
def make_room(xy_polygon, z0=0, z1=10):
    n = len(xy_polygon)
    bot = [Vertex.ByCoordinates(x, y, z0) for x, y in xy_polygon]
    top = [Vertex.ByCoordinates(x, y, z1) for x, y in xy_polygon]
    faces = []
    faces.append(Face.ByVertices(bot))
    faces.append(Face.ByVertices(list(reversed(top))))
    for i in range(n):
        j = (i + 1) % n
        faces.append(Face.ByVertices([bot[i], top[i], top[j], bot[j]]))
    return Cell.ByFaces(faces)


def box(x0, y0, x1, y1, z0=0, z1=10):
    xmn, xmx = min(x0, x1), max(x0, x1)
    ymn, ymx = min(y0, y1), max(y0, y1)
    return make_room([(xmn, ymn), (xmx, ymn), (xmx, ymx), (xmn, ymx)], z0, z1)

In [ ]:
rooms = [
    # N-S wing
    box(75, -7,  99, 17),                                                              # 0  Bedroom 1
    box(75, -43, 78, -7),                                                              # 1  Corridor
    make_room([(78,-7),(99,-7),(99,-22),(86,-22),(86,-19),(78,-19)]),                  # 2  Bedroom 2
    box(78, -25, 86, -19),                                                             # 3  WC 1
    box(78, -31, 86, -25),                                                             # 4  WC 2
    make_room([(78,-31),(78,-43),(99,-43),(99,-22),(86,-22),(86,-31)]),                # 5  Kitchen
    # E-W wing
    make_room([(35,-24),(60,-24),(60,-7),(75,-7),(75,0),(35,0)]),                      # 6  Living Room
    box(5, -17, 15, -7),                                                               # 7  WC
    make_room([(0,-17),(0,0),(35,0),(35,-24),(15,-24),(15,-7),(5,-7),(5,-17)]),        # 8  Office
    box(0, -24, 15, -17),                                                              # 9  Pantry
]

# Reference centroids for each room definition (order matches rooms[] above)
_ref_pts = [
    (87.0,   5.0, 5.0),   # Bedroom 1
    (76.5, -25.0, 5.0),   # Corridor
    (89.0, -14.0, 5.0),   # Bedroom 2
    (82.0, -22.0, 5.0),   # WC 1
    (82.0, -28.0, 5.0),   # WC 2
    (89.8, -33.7, 5.0),   # Kitchen
    (50.5, -10.7, 5.0),   # Living Room
    (10.0, -12.0, 5.0),   # WC
    (20.3, -10.6, 5.0),   # Office
    ( 7.5, -20.5, 5.0),   # Pantry
]
_room_names = ["Bedroom 1", "Corridor", "Bedroom 2", "WC 1", "WC 2",
               "Kitchen", "Living Room", "WC", "Office", "Pantry"]

def _dist3(a, b):
    return sum((x-y)**2 for x,y in zip(a,b))**0.5

cc = CellComplex.ByCells(rooms)
cells = Topology.Cells(cc)
print(f'{len(cells)} cells in CellComplex')

n = len(cells)
colors = ['#%02x%02x%02x' % tuple(int(c*255) for c in colorsys.hsv_to_rgb(i/n, 0.6, 0.9))
          for i in range(n)]

for i, cell in enumerate(cells):
    c  = Topology.Centroid(cell)
    cp = (c.X(), c.Y(), c.Z())
    idx  = min(range(len(_ref_pts)), key=lambda j: _dist3(cp, _ref_pts[j]))
    name = _room_names[idx]
    Topology.SetDictionary(cell, Dictionary.ByKeysValues(['color', 'id', 'name'], [colors[i], i, name]))
    print(f'  cell {i}: ({cp[0]:.1f}, {cp[1]:.1f}) → {name}')

Topology.Show(cc,
              faceColorKey='color', faceOpacity=0.5,
              edgeColor='white', edgeWidth=1,
              showVertices=False,
              backgroundColor='black',
              width=800, height=600,
              renderer=renderer)

10 cells in CellComplex
  cell 0: (83.5, 2.2) → Bedroom 1
  cell 1: (77.1, -25.0) → Corridor
  cell 2: (87.7, -16.0) → Bedroom 2
  cell 3: (56.7, -10.3) → Living Room
  cell 4: (87.4, -31.0) → Kitchen
  cell 5: (82.0, -28.0) → WC 2
  cell 6: (82.8, -22.0) → WC 1
  cell 7: (13.9, -12.6) → WC
  cell 8: (7.0, -19.8) → Pantry
  cell 9: (10.0, -12.0) → WC


In [ ]:
apertures = Topology.Faces(Cluster.ByTopologies(Topology.ByOBJPath(APTS)))
print(f'{len(apertures)} apertures loaded')
cc_apts = Topology.AddApertures(cc, apertures)
print('Done')

23 apertures loaded
Done


## 3. Primal vs Dual  (→ S02-01)

**Primal graph** — vertices at the geometric corners of the CellComplex, edges along wall boundaries.  
**Dual graph** — one vertex per room, an edge between every pair of rooms that share a face.

In [ ]:
# Primal: the geometric vertices and edges of the CellComplex ARE the primal graph
print(f'Primal — Vertices: {len(Topology.Vertices(cc))}, Edges: {len(Topology.Edges(cc))}')

Topology.Show(cc,
              faceColor='blue', faceOpacity=0.2,
              edgeColor='red', edgeWidth=2,
              vertexColor='red', vertexSize=10,
              showVertices=True,
              backgroundColor='black',
              width=800, height=600,
              renderer=renderer)

Primal — Vertices: 58, Edges: 105


In [ ]:
# Dual: one node per room, edges through shared walls
g_dual = Graph.ByTopology(cc_apts)
print(f'Dual — Nodes: {len(Graph.Vertices(g_dual))}, Edges: {len(Graph.Edges(g_dual))}')

for v in Graph.Vertices(g_dual):
    Topology.SetDictionary(v, Dictionary.ByKeysValues(['size', 'color'], [18, 'red']))
for e in Graph.Edges(g_dual):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(['width', 'color'], [4, 'white']))

Topology.Show(cc_apts, g_dual,
              faceColor='blue', faceOpacity=0.2,
              edgeColor='white', edgeWidth=0.5,
              vertexSizeKey='size', vertexColorKey='color',
              edgeWidthKey='width', edgeColorKey='color',
              showVertices=True,
              backgroundColor='black',
              width=800, height=600,
              renderer=renderer)

Dual — Nodes: 10, Edges: 16
